# Lab 6

You are tasked with evaluating card counting strategies for black jack. In order to do so, you will use object oriented programming to create a playable casino style black jack game where a computer dealer plays against $n$ computer players and possibly one human player. If you don't know the rules of blackjack or card counting, please google it. 

A few requirements:
* The game should utilize multiple 52-card decks. Typically the game is played with 6 decks.
* Players should have chips.
* Dealer's actions are predefined by rules of the game (typically hit on 16). 
* The players should be aware of all shown cards so that they can count cards.
* Each player could have a different strategy.
* The system should allow you to play large numbers of games, study the outcomes, and compare average winnings per hand rate for different strategies.

1. Begin by creating a classes to represent cards and decks. The deck should support more than one 52-card set. The deck should allow you to shuffle and draw cards. Include a "plastic" card, placed randomly in the deck. Later, when the plastic card is dealt, shuffle the cards before the next deal.

In [8]:
import random


# class with 52 cards, one extra plastic card, so 53 total
class Card:
  RANKS = ['2', '3', '4', '5', '6', '7', '8', '9', '10', 'J', 'Q', 'K', 'A']
  SUITS = ['Hearts', 'Diamonds', 'Clubs', 'Spades']

  def __init__(self, rank, suit, is_plastic=False):
    self.rank = rank
    self.suit = suit
    self.is_plastic = is_plastic

  def get_value(self):
    if self.is_plastic:
      return 0
    if self.rank in ['J', 'Q', 'K']:
      return 10
    if self.rank == 'A':
      return 11
    return int(self.rank)

  def __str__(self):
    if self.is_plastic:
      return 'PLASTIC CUT CARD'
    return f'{self.rank} of {self.suit}'

  def __repr__(self):
    return self.__str__()


# class of decks of cards. can support more than one 52-set of cards. able to shuffle and draw cards. shuffles when plastic card is drawn before next deal
class Deck:

  def __init__(self, num_decks=6):
    self.num_decks = num_decks
    self.cards = []
    self.needs_shuffle = False
    # automatically build and shuffle the shoe when Deck is created
    self.shuffle_deck()

  def create_deck(self):
    self.cards = []
    for _ in range(self.num_decks):
      for suit in Card.SUITS:
        for rank in Card.RANKS:
          self.cards.append(Card(rank, suit))

  def shuffle_deck(self):
    self.create_deck()
    random.shuffle(self.cards)

    # insert plastic card at a random position in shoe, of course not top or bottom
    plastic_card = Card('PLASTIC', 'PLASTIC', is_plastic=True)
    insert_position = random.randint(1, len(self.cards) - 1)
    self.cards.insert(insert_position, plastic_card)

    self.needs_shuffle = False

  def draw_card(self):
    # if plastic card is drawn flags needs_shuffle for next deal and draws the next real card

    if len(self.cards) == 0:
      self.shuffle_deck()

    drawn_card = self.cards.pop()

    if drawn_card.is_plastic:
      self.needs_shuffle = True
      # draw the next card so a player never receives the plastic card
      return self.draw_card()

    return drawn_card

In [ ]:
# test

if __name__ == '__main__':
  random.seed(8)  # fixed seed for reproducible test output
  deck = Deck(num_decks=6)
  print(
      f'Created {deck.num_decks}-deck shoe with {len(deck.cards)} total'
      f' cards ({len(deck.cards)-1} regular + 1 plastic cut card)\n'
  )

  # preview the first 5 drawn cards and their values
  print('First 5 cards drawn:')
  for i in range(1, 6):
    sample_card = deck.draw_card()
    print(f'  Card {i}: {sample_card} (Value: {sample_card.get_value()})')
  print()

  # reset shoe and simulate dealing 80 four-card rounds (320 cards total)
  deck.shuffle_deck()
  total_cards_dealt = 0
  plastic_triggered_count = 0

  for round_num in range(1, 81):
    # check before each new deal if the plastic card was drawn in the previous round
    if deck.needs_shuffle:
      print(
          f'[Before Round {round_num}] Reshuffling shoe because plastic card'
          ' was reached in the previous round'
      )
      deck.shuffle_deck()
      print(f'  -> Shoe restored to {len(deck.cards)} cards\n')

    # simulate dealing 4 cards in this round
    round_cards = []
    for _ in range(4):
      was_flagged = deck.needs_shuffle
      card = deck.draw_card()
      round_cards.append(card)
      total_cards_dealt += 1

      # verify that the plastic card was never dealt into a hand
      assert not card.is_plastic, 'Error: Plastic card was dealt to a player!'
      assert 2 <= card.get_value() <= 11, f'Error: Invalid card value on {card}'

      if not was_flagged and deck.needs_shuffle:
        plastic_triggered_count += 1
        print(
            f'[Round {round_num}] Plastic cut card encountered mid-round!'
            f' Dealt replacement card: {card}'
        )
        print(
            f'  -> Cards remaining in shoe: {len(deck.cards)} |'
            f' needs_shuffle: {deck.needs_shuffle}'
        )

  print(
      f'\nTest complete: {total_cards_dealt} real cards dealt across 80 rounds,'
      f' plastic cut card triggered {plastic_triggered_count} time(s), and 0'
      ' plastic cards ended up in a hand'
  )

Created 6-deck shoe with 313 total cards (312 regular + 1 plastic cut card)

First 5 cards drawn:
  Card 1: A of Hearts (Value: 11)
  Card 2: 9 of Clubs (Value: 9)
  Card 3: Q of Clubs (Value: 10)
  Card 4: A of Hearts (Value: 11)
  Card 5: 9 of Spades (Value: 9)

[Round 61] Plastic cut card encountered mid-round! Dealt replacement card: 6 of Clubs
  -> Cards remaining in shoe: 70 | needs_shuffle: True
[Before Round 62] Reshuffling shoe because plastic card was reached in the previous round
  -> Shoe restored to 313 cards


Test complete: 320 real cards dealt across 80 rounds, plastic cut card triggered 1 time(s), and 0 plastic cards ended up in a hand


2. Now design your game on a UML diagram. You may want to create classes to represent, players, a hand, and/or the game. As you work through the lab, update your UML diagram. At the end of the lab, submit your diagram (as pdf file) along with your notebook. 

3. Begin with implementing the skeleton (ie define data members and methods/functions, but do not code the logic) of the classes in your UML diagram.

In [29]:
class Hand:

  def __init__(self):
    self.cards: list[Card] = []
    self.bet: int = 0

  def add_card(self, card: Card) -> None:
    pass

  def get_total(self) -> int:
    pass

  def is_bust(self) -> bool:
    pass

  def is_blackjack(self) -> bool:
    pass

  def clear(self) -> None:
    pass


class Player:

  def __init__(self, name: str = 'Player', chips: int = 1000):
    self.name: str = name
    self.chips: int = chips
    self.hand: Hand = Hand()
    self.hands_played: int = 0
    self.net_winnings: int = 0

  def place_bet(self, seen_cards: list[Card]) -> int:
    pass

  def decide_action(self, dealer_upcard: Card, seen_cards: list[Card]) -> str:
    pass

  def win_bet(self, multiplier: float) -> None:
    pass


class Dealer(Player):

  def __init__(self):
    super().__init__(name='Dealer', chips=0)

  def decide_action(self, dealer_upcard: Card, seen_cards: list[Card]) -> str:
    pass


class HumanPlayer(Player):

  def __init__(self, name: str = 'Human', chips: int = 1000):
    super().__init__(name=name, chips=chips)

  def place_bet(self, seen_cards: list[Card]) -> int:
    pass

  def decide_action(self, dealer_upcard: Card, seen_cards: list[Card]) -> str:
    pass


class AIPlayer(Player):

  def __init__(
      self,
      name: str = 'AI',
      chips: int = 1000,
      strategy_type: str = 'basic',
  ):
    super().__init__(name=name, chips=chips)
    self.strategy_type: str = strategy_type

  def place_bet(self, seen_cards: list[Card]) -> int:
    pass

  def decide_action(self, dealer_upcard: Card, seen_cards: list[Card]) -> str:
    pass

  def count_cards(self, seen_cards: list[Card]) -> int:
    pass


class BlackjackGame:

  def __init__(
      self,
      players: list[Player] = None,
      num_decks: int = 6,
  ):
    self.deck: Deck = Deck(num_decks=num_decks)
    self.dealer: Dealer = Dealer()
    self.players: list[Player] = players if players is not None else []
    self.seen_cards: list[Card] = []

  def deal_initial_cards(self) -> None:
    pass

  def play_round(self) -> None:
    pass

  def check_win(self) -> None:
    pass

  def display(self) -> None:
    pass

  def run_simulation(self, num_games: int) -> dict:
    pass

4. Complete the implementation by coding the logic of all functions. For now, just implement the dealer player and human player.

5.  Test. Demonstrate game play. For example, create a game of several dealer players and show that the game is functional through several rounds.

6. Implement a new player with the following strategy:

    * Assign each card a value: 
        * Cards 2 to 6 are +1 
        * Cards 7 to 9 are 0 
        * Cards 10 through Ace are -1
    * Compute the sum of the values for all cards seen so far.
    * Hit if sum is very negative, stay if sum is very positive. Select a threshold for hit/stay, e.g. 0 or -2.  

7. Create a test scenario where one player, using the above strategy, is playing with a dealer and 3 other players that follow the dealer's strategy. Each player starts with same number of chips. Play 50 rounds (or until the strategy player is out of money). Compute the strategy player's winnings. You may remove unnecessary printouts from your code (perhaps implement a verbose/quiet mode) to reduce the output.

8. Create a loop that runs 100 games of 50 rounds, as setup in previous question, and store the strategy player's chips at the end of the game (aka "winnings") in a list. Histogram the winnings. What is the average winnings per round? What is the standard deviation. What is the probability of net winning or losing after 50 rounds?


9. Repeat previous questions scanning the value of the threshold. Try at least 5 different threshold values. Can you find an optimal value?

10. Create a new strategy based on web searches or your own ideas. Demonstrate that the new strategy will result in increased or decreased winnings. 